
# Lab: Data Cleaning and Reporting

## Objectives
1. Understand common data quality issues and map them to Quality Dimensions (Accuracy, Completeness, Consistency, Validity, Uniqueness).
2. Learn how to audit data using Pandas.
3. Practice cleaning techniques
4. **Important**: Log every cleaning step to a database for reproducibility and reporting.

**Note**: In this lab, we will create a new DataFrame variable for each major step to preserve the state of the data at each stage.


In [59]:

# Install necessary libraries
!pip install pandas numpy matplotlib seaborn



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [60]:

import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Database connection
db_name = 'customers.db'
conn = sqlite3.connect(db_name)

# Helper function to log cleaning steps
def log_cleaning_step(step_num, action, col, details, reason, rows_affected):
    cursor = conn.cursor()
    
    # Check if this step already exists
    cursor.execute("SELECT count(*) FROM cleaning_logs WHERE step_number = ?", (step_num,))
    exists = cursor.fetchone()[0]
    
    if exists > 0:
        # Option A: Update the existing log (if you re-run with different logic)
        cursor.execute('''
            UPDATE cleaning_logs 
            SET action_type = ?, column_affected = ?, details = ?, reason = ?, rows_affected = ?, timestamp = CURRENT_TIMESTAMP
            WHERE step_number = ?
        ''', (action, col, details, reason, rows_affected, step_num))
        print(f"Log updated: Step {step_num} - {action} on {col}")
    else:
        # Option B: Insert new log
        cursor.execute('''
            INSERT INTO cleaning_logs (step_number, action_type, column_affected, details, reason, rows_affected)
            VALUES (?, ?, ?, ?, ?, ?)
        ''', (step_num, action, col, details, reason, rows_affected))
        print(f"Log added: Step {step_num} - {action} on {col}")
        
    conn.commit()



## 1. Data Loading & Initial Audit
First, we load the raw data and check its general health.


In [61]:

df = pd.read_sql_query("SELECT * FROM customers", conn)
print(f"Initial Shape: {df.shape}")
df.head()


Initial Shape: (210, 7)


,cust_id,full_name,email,age,income,join_date,city
0,C1001,User 1,user1@example.com,17,32066.34,2023-12-08,london
1,C1002,User 2,user2@example.com,19,143018.16,2023-11-22,None
2,C1003,User 3,user3@example.com,10,29415.49,2023-06-21,new york
3,C1004,User 4,user4@example.com,36,72452.85,2023-10-22,None
4,C1005,User 5,USER5@EXAMPLE.COM,8,59177.94,2023-02-12,new york


In [62]:

# Check data types and missing values
# Note: 'Non-Null Count' shows how many values are NOT missing.
# The 'Dtype' column shows the data type. 'object' usually means string/text.
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 210 entries, 0 to 209
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   cust_id    210 non-null    object
 1   full_name  210 non-null    object
 2   email      184 non-null    object
 3   age        210 non-null    object
 4   income     189 non-null    object
 5   join_date  210 non-null    object
 6   city       159 non-null    object
dtypes: object(7)
memory usage: 11.6+ KB



**Observations:**
- `age` and `income` are objects (strings), but should be numeric.
- `email`, `city`, `income` have missing values.
- `cust_id` might have duplicates.
- `join_date` is object (string), needs to be date type.

**We found Age, Income as a string/object type. Why is this a problem?**

You cannot perform mathematical operations (mean, max, correlation) on strings. It breaks analysis pipelines.



## 2. Dimension: Uniqueness (Handling Duplicates)
Duplicate records can skew analysis. We check for exact duplicates.


In [63]:

# Check duplicates
duplicates = df.duplicated().sum()
print(f"Duplicate rows found: {duplicates}")

# Show sample duplicates (Sorted to show pairs)
if duplicates > 0:
    print("Sample duplicate rows (Original + Duplicate):")
    # Sorting by cust_id ensures the original and its copy appear next to each other
    # We use keep=False to mark ALL duplicates as True
    display(df[df.duplicated(keep=False)].sort_values(by='cust_id').head(6))


Duplicate rows found: 10
Sample duplicate rows (Original + Duplicate):


,cust_id,full_name,email,age,income,join_date,city
0,C1001,User 1,user1@example.com,17,32066.34,2023-12-08,london
200,C1001,User 1,user1@example.com,17,32066.34,2023-12-08,london
1,C1002,User 2,user2@example.com,19,143018.16,2023-11-22,None
201,C1002,User 2,user2@example.com,19,143018.16,2023-11-22,None
2,C1003,User 3,user3@example.com,10,29415.49,2023-06-21,new york
202,C1003,User 3,user3@example.com,10,29415.49,2023-06-21,new york


In [64]:
# Remove duplicates (Create new DF)
df_deduped = df.drop_duplicates().copy()
current_count = len(df_deduped)
initial_count = len(df)
rows_dropped = initial_count - current_count

# Log the step
reason = "Exact duplicates provide no new information and can bias results (e.g., inflating counts)."
log_cleaning_step(step_num=1, action='Removal', col='All', details='Removed duplicate records', reason=reason, rows_affected=int(rows_dropped))


Log updated: Step 1 - Removal on All


In [65]:
# Test
assert df_deduped.duplicated().sum() == 0
print(f"New Shape: {df_deduped.shape}")

New Shape: (200, 7)



## 3. Dimension: Consistency 


### 1) Handline Type Consistency & Schema Issues
`Age`, `Income` should be numbers. `join_date` should be a date.
Values that cannot be converted (like "unknown") will become NaN (Null).

In [66]:

# Audit: Check for non-numeric values
# Instead of guessing patterns, let's find values that FAIL to convert to numbers.

# For Age:
print("--- Unique non-numeric Age values ---")
age_numeric = pd.to_numeric(df_deduped['age'], errors='coerce')
# Find indices where result is NaN but original was NOT Null
problematic_age_mask = age_numeric.isnull() & df_deduped['age'].notnull()
print(df_deduped.loc[problematic_age_mask, 'age'].unique())

# For Income:
print("\n--- Sample of non-numeric Income values ---")
income_numeric = pd.to_numeric(df_deduped['income'], errors='coerce')
problematic_income_mask = income_numeric.isnull() & df_deduped['income'].notnull()
# Show a sample of these problematic values
print(df_deduped.loc[problematic_income_mask, 'income'].head(10).values)


--- Unique non-numeric Age values ---
['8 years' 'unknown' '15 years' '10 years' '17 years' '18 years'
 '23 years']

--- Sample of non-numeric Income values ---
['$10074.14' '$94033.84' '$144962.87' '$9728.71' '$42880.41' '$200208.32'
 '$78492.54' '$71902.93' '$14444.24' '$61665.84']


In [67]:

# Create new DF for Type Cleaning
df_types = df_deduped.copy()

# Check Nulls BEFORE coercion
print("Nulls before coercion:")
print(df_types[['age', 'income']].isnull().sum())

# Coerce Age
# We extract numbers first for better handling of "27 years"
df_types['age_updated'] = df_types['age'].astype(str).str.extract(r'(\d+)', expand=False)
df_types['age_updated'] = pd.to_numeric(df_types['age_updated'], errors='coerce')

# Coerce Income
# 1. Handle Dollar Signs (Remove them)
df_types['income'] = df_types['income'].astype(str).str.replace('$', '', regex=False)

# 2. Convert to Numeric
df_types['income'] = pd.to_numeric(df_types['income'], errors='coerce')

# Check Nulls AFTER coercion
print("\nNulls after coercion (Notice the increase due to invalid formats being turned to NaN):")
print(df_types[['age_updated', 'income']].isnull().sum())

# Log
age_changed = (df_types['age'] != df_deduped['age'])
income_changed = (df_types['income'] != df_deduped['income'])
rows_affected_type_casting = (age_changed | income_changed).sum()
reason = "Calculations require numeric types. 'Coerce' handles mixed formats/errors by converting to NaN."
log_cleaning_step(step_num=2, action='Type Casting', col='age, income', details='Converted to numeric, invalid values set to NaN', reason=reason, rows_affected=int(rows_affected_type_casting))


Nulls before coercion:
age        0
income    20
dtype: int64

Nulls after coercion (Notice the increase due to invalid formats being turned to NaN):
age_updated     8
income         20
dtype: int64
Log updated: Step 2 - Type Casting on age, income


In [68]:
# Show a sample of these null values
problematic_age_mask = df_types['age_updated'].isnull() & df_types['age'].notnull()
print(df_types.loc[problematic_age_mask, 'age'].values)

['unknown' 'unknown' 'unknown' 'unknown' 'unknown' 'unknown' 'unknown'
 'unknown']


### Join Date Consistency

In [69]:
# For Join date:
print("\n--- Sample of Join Date values ---")
print(df_deduped['join_date'].unique())


--- Sample of Join Date values ---
['2023-12-08' '2023-11-22' '2023-06-21' '2023-10-22' '2023-02-12'
 '2023-10-04' '2023-07-26' '2023-10-12' '2023-11-20' '2023-04-20'
 '2023-08-14' '2023-10-29' '2023-11-05' '2023-10-26' '2023-04-01'
 '2023-08-27' '16/09/2023' '2023-05-04' '2023-07-13' '2023-05-27'
 '2023-01-08' '2023-04-04' '2023-08-13' '2026-07-25' '2023-03-04'
 '2023-02-05' '2023-04-11' '2023-08-06' 'not_available' '2023-09-29'
 '2023-05-28' '2023-08-07' '2023-06-12' '14/06/2023' '2023-10-28'
 '2023-07-25' '2023-03-12' '2023-11-27' '2023-08-02' '2023-10-24'
 '2023-02-22' '2023-09-24' '2023-07-12' '2023-07-01' '2023-07-06'
 '2023-03-15' '2023-04-05' '2026-04-29' '2023-02-28' '2023-08-01'
 '2023-10-16' '30/09/2023' '2023-08-18' '2023-10-15' '2023-05-10'
 '2023-09-18' '2023-02-24' '2023-09-23' '2023-01-21' '2023-11-14'
 '2023-02-17' '2023-09-19' '2023-02-21' '2023-12-24' '26/11/2023'
 '2023-05-17' '2023-06-13' '2026-08-17' '2023-10-20' '2023-09-08'
 '2023-08-28' '2023-07-11' '2023-01-2

In [70]:
# Coerce Date
# Use errors='coerce' to handle 'not_available' or bad formats
# 1. Try parsing as ISO format (YYYY-MM-DD) - Standard
date_iso = pd.to_datetime(df_types['join_date'], format='%Y-%m-%d', errors='coerce')

# 2. Try parsing as European format (DD/MM/YYYY) - The 'problematic' ones
date_euro = pd.to_datetime(df_types['join_date'], format='%d/%m/%Y', errors='coerce')

# 3. Combine: Use ISO result, but fill failures (NaT) with European result
df_types['join_date_updated'] = date_iso.fillna(date_euro)

In [71]:
# Check Nulls BEFORE coercion
print("Nulls before coercion:")
print(df_types[['join_date']].isnull().sum())
date_nulls_before = df_types['join_date'].isnull().sum()
# Check Nulls AFTER coercion
print("\nNulls after coercion (Notice the increase due to invalid formats being turned to NaN):")
print(df_types[['join_date_updated']].isnull().sum())
date_nulls_after = df_types['join_date_updated'].isnull().sum()

# Log
join_date_changed = (df_types['join_date'] != df_deduped['join_date'])
rows_affected_date = income_changed.sum()
reason = " Dates need datetime format for time-based logic. "
log_cleaning_step(step_num=3, action='Type Casting', col='join_date', details='Converted to datetime, invalid values set to NaN', reason=reason, rows_affected=int(rows_affected_date))


Nulls before coercion:
join_date    0
dtype: int64

Nulls after coercion (Notice the increase due to invalid formats being turned to NaN):
join_date_updated    5
dtype: int64
Log updated: Step 3 - Type Casting on join_date


In [72]:
# Show a sample of these null values
problematic_join_date_mask = df_types['join_date_updated'].isnull() & df_types['join_date'].notnull()
print(df_types.loc[problematic_join_date_mask, 'join_date'].values)

['not_available' 'not_available' 'not_available' 'not_available'
 'not_available']


In [73]:
df_types['join_date']=df_types['join_date_updated']
df_types['age']=df_types['age_updated']
df_types = df_types.drop(columns=['join_date_updated', 'age_updated'])


### 2) Standardization
Check `city` and `email` for casing issues and synonyms (e.g., "NY" vs "New York").


In [74]:

print("--- City Value Counts (Before) ---")
print(df_types['city'].unique())

print("--- Email Value Counts (Before) ---")
print(df_types['email'].unique())


--- City Value Counts (Before) ---
['london' None 'new york' 'Paris' 'NY' 'CAIRO' 'Cairo' 'London' 'New York']
--- Email Value Counts (Before) ---
['user1@example.com' 'user2@example.com' 'user3@example.com'
 'user4@example.com' 'USER5@EXAMPLE.COM' 'user6@example.com'
 'user7@example.com' None 'user9@example.com' 'USER10@EXAMPLE.COM'
 'user11@example.com' 'user12@example.com' 'user13@example.com'
 'user14@example.com' 'USER15@EXAMPLE.COM' 'user17@example.com'
 'user18@example.com' 'user19@example.com' 'USER20@EXAMPLE.COM'
 'user21@example.com' 'user22@example.com' 'user23@example.com'
 'USER25@EXAMPLE.COM' 'user26@example.com' 'user27@example.com'
 'user28@example.com' 'user29@example.com' 'USER30@EXAMPLE.COM'
 'user31@example.com' 'user33@example.com' 'user34@example.com'
 'USER35@EXAMPLE.COM' 'user36@example.com' 'user37@example.com'
 'user38@example.com' 'user39@example.com' 'user41@example.com'
 'user42@example.com' 'user43@example.com' 'user44@example.com'
 'USER45@EXAMPLE.COM' 'u

In [75]:
city_before = df_types['city'].copy()
email_before = df_types['email'].copy()

# Create new DF for Standardization
df_standardized = df_types.copy()

# Standardize City
# 1. Lowercase and strip whitespace
df_standardized['city'] = df_standardized['city'].str.lower().str.strip()

# 2. Map abbreviations (ny -> new york)
df_standardized['city'] = df_standardized['city'].replace({'ny': 'new york'})

# Standardize Email
df_standardized['email'] = df_standardized['email'].str.lower().str.strip()

city_changed = (df_standardized['city'].fillna('') != city_before.fillna(''))
email_changed = (df_standardized['email'].fillna('') != email_before.fillna(''))
rows_affected_std = (city_changed | email_changed).sum()
reason = "Inconsistent casing (New York vs new york) and abbreviations (NY) create false distinct categories. Normalizing ensures accurate grouping."
log_cleaning_step(step_num=4, action='Standardization', col='city, email', details='Lowercased, stripped whitespace, mapped NY->new york', reason=reason, rows_affected=int(rows_affected_std))

print("\n--- City Value Counts (After) ---")
print(df_standardized['city'].unique())


Log updated: Step 4 - Standardization on city, email

--- City Value Counts (After) ---
['london' None 'new york' 'paris' 'cairo']


In [76]:

logs = pd.read_sql_query("SELECT * FROM cleaning_logs ORDER BY step_number", conn)
logs

,log_id,step_number,action_type,column_affected,details,reason,rows_affected,timestamp
0,1,1,Removal,All,Removed duplicate records,Exact duplicates provide no new information an...,10,2026-03-07 12:02:07
1,2,2,Type Casting,"age, income","Converted to numeric, invalid values set to NaN",Calculations require numeric types. 'Coerce' h...,200,2026-03-07 12:02:08
2,3,3,Type Casting,join_date,"Converted to datetime, invalid values set to NaN",Dates need datetime format for time-based log...,200,2026-03-07 12:02:08
3,4,4,Standardization,"city, email","Lowercased, stripped whitespace, mapped NY->ne...",Inconsistent casing (New York vs new york) and...,130,2026-03-07 12:02:08


In [77]:
df_standardized.to_sql('partially_cleaned_customers', conn, if_exists='replace', index=False)
print(f"Successfully saved {len(df_standardized)} rows to 'customers.db' in table 'partially_cleaned_customers'.")
conn.close()


Successfully saved 200 rows to 'customers.db' in table 'partially_cleaned_customers'.
